In [6]:
#!pip install autots

In [7]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from autots import AutoTS

In [8]:
# 1. AYARLAR VE COIN LİSTESİ
# İstediğin kadar ekleme yapabilirsin
coin_list = ["BTC-USD", "ETH-USD", "DOGE-USD", "SOL-USD", "AVAX-USD"]
forecast_days = 7 # Kaç günlük tahmin istiyoruz?

In [ ]:
def analyze_coins(tickers):
    results = {}
    
    # Grafik alanı oluşturma (Coin sayısı kadar satır)
    fig, axes = plt.subplots(len(tickers), 1, figsize=(12, 5 * len(tickers)))
    if len(tickers) == 1: axes = [axes] # Tek coin varsa liste yapısına sok

    for i, ticker in enumerate(tickers):
        print(f"\n🚀 {ticker} için analiz başlatılıyor...")
        
        # Veri Çekme
        data = yf.download(ticker, period="1y", interval="1d")
        data = data.reset_index()
        
        # AutoTS Model Yapılandırması
        model = AutoTS(
            forecast_length=forecast_days,
            frequency='infer',
            prediction_interval=0.9,
            ensemble='simple',
            model_list="fast",     # 'superfast' yaparsan daha da hızlanır
            max_generations=5,
            num_validations=2,
            verbose=-1,             # <--- Hataları gizlemek için -1 veya 0 yapabilirsin
            drop_data_older_than_periods=200
        )        
        model = model.fit(data, date_col='Date', value_col='Close')
        prediction = model.predict()
        forecast = prediction.forecast
        
        # Sonuçları Kaydet
        results[ticker] = forecast
        
        # --- GÖRSELLEŞTİRME ---
        # Geçmiş veri (Son 30 gün)
        recent_data = data.tail(30)
        axes[i].plot(recent_data['Date'], recent_data['Close'], label='Gerçek Fiyat', marker='o')
        
        # Tahmin verisi
        axes[i].plot(forecast.index, forecast.values, label='Tahmin', color='red', linestyle='--', marker='s')
        
        # Güven Aralığı (Tahmin belirsizliğini boyar)
        axes[i].fill_between(
            prediction.lower_forecast.index,
            prediction.lower_forecast.iloc[:, 0],
            prediction.upper_forecast.iloc[:, 0],
            color='r', alpha=0.1, label='Güven Aralığı'
        )
        
        axes[i].set_title(f"{ticker} - {forecast_days} Günlük Projeksiyon")
        axes[i].legend()
        axes[i].set_ylabel("USD Fiyat")

    plt.tight_layout()
    plt.show()
    return results


In [ ]:
# Analizi başlat
predictions = analyze_coins(coin_list)